# Network Intrusion Detection Using Machine Learning

This notebook uses the shared project modules so the notebook and command-line workflow stay consistent. Place `kddcup.data.corrected` at `/content/kddcup.data.corrected` (Colab) or update `DATA_PATH` below. The experiment samples exactly 12,000 rows (seed 42); no result values are hard-coded.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_PATH = '/content/kddcup.data.corrected'
# For a local checkout, use: DATA_PATH = str(PROJECT_ROOT / 'data' / 'kddcup.data.corrected')

## Load and inspect
The final field is the attack label. All non-normal labels become the positive `attack` class. Invalid numeric values are coerced to missing and imputed inside training folds.

In [ ]:
from data_preprocessing import load_sampled_data, clean_data, describe_data
raw = load_sampled_data(DATA_PATH)
describe_data(raw)
data = clean_data(raw)
print('Feature matrix:', data.drop(columns='target').shape)
print(data['target'].value_counts().rename(index={0: 'normal', 1: 'attack'}))

## Split, preprocessing, models, and evaluation
The shared training function makes a stratified holdout split before any learned preprocessing. Imputation, one-hot encoding, and scaling live inside sklearn pipelines. Random Forest and SVC are tuned with training-only cross-validation; Isolation Forest and One-Class SVM learn without labels. All four are evaluated against the held-out labels. The old notebook also used an autoencoder with a threshold selected from test errors; that threshold leaks test information, so this revised workflow leaves it out.

In [ ]:
from train import run
metrics = run(DATA_PATH, output_dir=str(PROJECT_ROOT))
metrics

## Reuse the saved model
The fitted artifact includes the same learned preprocessing stages. Provide raw KDD field names (categorical values such as `tcp`, `http`, and `sf` are accepted).

In [ ]:
from predict import predict_network_traffic
# Replace or extend this partial example with a real connection record. Missing feature
# fields are imputed from training data.
example = {'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'sf', 'src_bytes': 181, 'dst_bytes': 5450}
predict_network_traffic(example, PROJECT_ROOT / 'models' / 'random_forest.pkl')